# B 담당 Colab 실행 노트북 (딥 특징 추출 + 평균점 섞기)

실행 전용 노트북이다. 실험 로직은 `src/`의 `.py`에만 있고, 여기서는 환경 준비 · 스크립트 실행 · 결과 훑어보기만 한다.

- 런타임 유형을 **T4 GPU**로 먼저 바꾼 뒤 시작한다(중간에 바꾸면 `/content`가 비워져 데이터를 다시 복사해야 한다).
- 세션을 다시 열면 **셀 1~6을 다시 실행**하고 셀 7(진행 상태)부터 본다. 끝나지 않은 셀은 그대로 다시 실행하면 이어서 진행한다.
- test 분할은 D8 전까지 쓰지 않는다. 평균점 섞기는 `--split dev`만 실행한다.
- 커밋 전에 출력을 지운다.

**셀 1. 설정 (채울 것)**: 저장소 주소, Drive 바로가기 경로 두 개를 채운다.

In [1]:
REPO_URL = "https://github.com/kimminkyung05/puppy.."
DRIVE_DIR = "/content/drive/MyDrive/puppy..."
DATA_DIR  = "/content/drive/MyDrive/Tsinghua Dogs Dataset"
DATA_ZIPS = []   # 비워 두면 셀 3이 저해상도 이미지·주석 zip을 자동으로 찾음
LOCAL_DATA = "/content/data"
REPO = "/content/dog-mix-cv"

**셀 2. Drive 연결, 폴더 확인**: 두 폴더가 보이는지 확인하고 데이터 폴더의 파일 목록을 출력한다.

In [2]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')
for d in [DRIVE_DIR, DATA_DIR]:
    assert os.path.isdir(d), (f"폴더가 없음: {d}\n"
        "→ 링크를 열고 '바로가기 추가 → 내 드라이브'를 했는지, 아래 목록에서 이름을 확인")
print("내 드라이브:", os.listdir("/content/drive/MyDrive"))
print("\n데이터 폴더 내용:")
for p in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True))[:100]:
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:10.1f} MB  {os.path.relpath(p, DATA_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
내 드라이브: ['Colab Notebooks', 'GoodNotes', '2026-1', '25', 'FinGuard', '이상치탐지.gsheet', 'week2-so101실행영상', 'eng', '증명서', 'ppt', 'paper', 'Tsinghua Dogs Dataset', 'Lecture 15-16_Offline_RL.pptx', 'puppy...']

데이터 폴더 내용:
       0.2 MB  TrainValSplit.zip
       2.8 MB  TrainValSplit/TrainAndValList/train.lst
       0.2 MB  TrainValSplit/TrainAndValList/validation.lst
      37.3 MB  low-annotations.zip
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100001.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100002.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100003.jpeg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100004.jpg.xml
       0.0 MB  low-annotations/Low-Annotations/1043-n000001-Shiba_Dog/n100005.jpg.xml
       0.0 MB  low-

In [17]:
import os

print(f"검증할 DRIVE_DIR: {DRIVE_DIR}")
if os.path.exists(DRIVE_DIR):
    print("[V] DRIVE_DIR 폴더가 정상적으로 존재합니다.")
    print("\n--- 하위 폴더 및 파일 목록 ---")
    try:
        sub_items = os.listdir(DRIVE_DIR)
        for item in sub_items:
            item_path = os.path.join(DRIVE_DIR, item)
            is_dir = "[폴더]" if os.path.isdir(item_path) else "[파일]"
            print(f"{is_dir} {item}")
            # evalset 폴더인 경우 내부 파일도 추가 출력
            if item == "evalset" and os.path.isdir(item_path):
                print("   └─ 하위 파일:", os.listdir(item_path))
    except Exception as e:
         print(f"[X] 하위 목록을 읽는 중 오류 발생: {e}")
else:
    print("[X] DRIVE_DIR 폴더가 존재하지 않습니다! 경로를 다시 확인해 주세요.")

검증할 DRIVE_DIR: /content/drive/MyDrive/puppy...
[V] DRIVE_DIR 폴더가 정상적으로 존재합니다.

--- 하위 폴더 및 파일 목록 ---
[폴더] features
[폴더] logs
[폴더] evalset
   └─ 하위 파일: ['raw', 'uf_draft', 'val_split_seed1.csv', 'eval_index_UF_panels_colab.gsheet']
[폴더] preds


**셀 3. 받을 zip 고르기**: 저해상도 이미지 · 주석 · 분할 zip만 고른다(고해상도 제외).

In [3]:
if not DATA_ZIPS:
    zips = glob.glob(f"{DATA_DIR}/**/*.zip", recursive=True)
    DATA_ZIPS = sorted(
        z for z in zips
        if "high" not in os.path.basename(z).lower()
        and any(k in os.path.basename(z).lower() for k in ["low", "annotation", "split", "trainval"])
    )
print("복사할 zip:", *DATA_ZIPS, sep="\n  ")
assert DATA_ZIPS, "자동으로 못 찾음 → 셀 2 출력을 보고 셀 1의 DATA_ZIPS에 경로를 직접 적기"

복사할 zip:
  /content/drive/MyDrive/Tsinghua Dogs Dataset/TrainValSplit.zip
  /content/drive/MyDrive/Tsinghua Dogs Dataset/low-annotations.zip
  /content/drive/MyDrive/Tsinghua Dogs Dataset/low-resolution.zip


**셀 4. `/content`로 복사 · 압축 해제**: 이번 세션에 이미 푼 zip은 건너뛴다. 마지막 줄에 풀린 폴더 이름이 나온다.

In [4]:
import shutil, subprocess
os.makedirs(LOCAL_DATA, exist_ok=True)
for z in DATA_ZIPS:
    name = os.path.basename(z)
    done = f"{LOCAL_DATA}/.done_{name}"
    if os.path.exists(done):
        print("이미 풀림:", name); continue
    try:
        print("복사:", name); shutil.copy(z, f"/content/{name}")
    except OSError as e:
        raise RuntimeError(f"{name} 복사 실패: {e}\n"
            "→ 공개 폴더 다운로드 한도일 수 있음. 그 zip을 '사본 만들기'로 내 드라이브에 복사한 뒤 "
            "셀 1의 DATA_ZIPS에 그 경로를 적기") from e
    print("압축 해제:", name)
    subprocess.run(["unzip", "-q", "-o", f"/content/{name}", "-d", LOCAL_DATA], check=True)
    os.remove(f"/content/{name}"); open(done, "w").close()
print(os.listdir(LOCAL_DATA))

이미 풀림: TrainValSplit.zip
이미 풀림: low-annotations.zip
이미 풀림: low-resolution.zip
['low-resolution', '.done_low-resolution.zip', 'Low-Annotations', '.done_low-annotations.zip', 'TrainValSplit', '.done_TrainValSplit.zip']


**셀 5. 코드 받기 + 경로 자동 설정**: 저장소를 clone/pull하고, 풀린 폴더를 찾아 `/content/paths_colab.yaml`을 만들어 `DOGMIX_PATHS`로 지정한다(`configs/paths.yaml`은 고치지 않는다).

In [5]:
import re, yaml
CLASS_RE = re.compile(r"^\d+-n\d+-")
def find_class_root(base, want_xml):
    for dirpath, dirnames, _ in os.walk(base):
        cls = [d for d in dirnames if CLASS_RE.match(d)]
        if len(cls) >= 100:
            sample = os.listdir(os.path.join(dirpath, cls[0]))[:50]
            if any(f.endswith(".xml") for f in sample) == want_xml:
                return dirpath
    return None
def find_file(base, name):
    for dirpath, _, files in os.walk(base):
        if name in files:
            return os.path.join(dirpath, name)
    return None
img_root = find_class_root(LOCAL_DATA, want_xml=False)
ann_root = find_class_root(LOCAL_DATA, want_xml=True)
train_lst, val_lst = find_file(LOCAL_DATA, "train.lst"), find_file(LOCAL_DATA, "validation.lst")
if not (train_lst and val_lst):
    !git clone -q https://github.com/dejungle/Tsinghua-Dogs /content/tsinghua-split
    train_lst = "/content/tsinghua-split/split/train.lst"
    val_lst   = "/content/tsinghua-split/split/validation.lst"
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
cfg = yaml.safe_load(open("configs/paths.yaml", encoding="utf-8"))
cfg["tsinghua"].update(image_root=img_root, annotation_root=ann_root, train_list=train_lst, val_list=val_lst)
cfg["evalset"]["index_csv"] = f"{DRIVE_DIR}/evalset/eval_index.csv"
cfg["outputs"].update(features_root=f"{DRIVE_DIR}/features", preds_root=f"{DRIVE_DIR}/preds", logs_root=f"{DRIVE_DIR}/logs")
with open("/content/paths_colab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)
os.environ["DOGMIX_PATHS"] = "/content/paths_colab.yaml"
print(open("/content/paths_colab.yaml", encoding="utf-8").read())
missing = [k for k, v in {"이미지": img_root, "주석": ann_root, "train.lst": train_lst, "validation.lst": val_lst}.items()
           if not v or not os.path.exists(v)]
assert not missing, f"못 찾음: {missing} → 셀 4의 폴더 목록을 보고 paths_colab.yaml을 직접 고치기"

/content/dog-mix-cv
classes:
  exclude:
  - chinese_rural_dog
  - dhole
  - African_hunting_dog
  exclude_dingo: false
evalset:
  index_csv: /content/drive/MyDrive/puppy.../evalset/eval_index.csv
outputs:
  features_root: /content/drive/MyDrive/puppy.../features
  logs_root: /content/drive/MyDrive/puppy.../logs
  preds_root: /content/drive/MyDrive/puppy.../preds
tsinghua:
  annotation_root: /content/data/Low-Annotations
  image_root: /content/data/low-resolution
  train_list: /content/data/TrainValSplit/TrainAndValList/train.lst
  val_list: /content/data/TrainValSplit/TrainAndValList/validation.lst



**셀 6. GPU 확인, 패키지 설치, Hugging Face 로그인**: DINOv3 접근 승인을 받은 계정의 토큰으로 로그인한다.

In [6]:
!nvidia-smi
!pip install -q -r requirements.txt
from huggingface_hub import login; login()

Sat Nov 21 17:19:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**셀 6-1. 데이터 · 모델 점검**: 경로, `bodybndbox` 태그, 몸통 박스를 못 찾은 비율, 클래스 수(127), 모델 ID, 출력 차원, DINOv3 `num_register_tokens`를 출력한다. '확인 필요'가 있으면 추출 전에 해결한다.

In [7]:
!python -m src.common.check_data
!python -m src.features.check_models

== 1) 경로가 실제로 있는지 ==
  설정 파일: /content/paths_colab.yaml
  [있음] tsinghua.image_root: /content/data/low-resolution
  [있음] tsinghua.annotation_root: /content/data/Low-Annotations
  [있음] tsinghua.train_list: /content/data/TrainValSplit/TrainAndValList/train.lst
  [있음] tsinghua.val_list: /content/data/TrainValSplit/TrainAndValList/validation.lst
  [없음] evalset.index_csv: /content/drive/MyDrive/puppy.../evalset/eval_index.csv
  [있음] outputs.features_root: /content/drive/MyDrive/puppy.../features
  [있음] outputs.logs_root: /content/drive/MyDrive/puppy.../logs
  [있음] outputs.preds_root: /content/drive/MyDrive/puppy.../preds

== 2) 분할 파일과 학습 클래스 수 ==
  train 분할 파일: 65228줄 → 제외 후 61592장
  val 분할 파일:   5200줄 → 제외 후 5080장
  학습 클래스: 제외 전 130개 → 제외 후 127개 (기대값 127): OK
  제외 클래스 'chinese_rural_dog': 목록에 있음 → 제외됨
  제외 클래스 'dhole': 목록에 있음 → 제외됨
  제외 클래스 'African_hunting_dog': 목록에 있음 → 제외됨
  dingo: 목록에 있음, exclude_dingo=False

== 3) 주석 xml의 bodybndbox ==
  점검 2000장 (학습셋 61592장 중)
  이미지 파일 없음: 0장
  xml 파일

**셀 7. 진행 상태 확인**: 세션을 다시 열면 셀 1~6 실행 후 여기부터 보고, '완료'가 아닌 것을 이어서 실행한다.

In [8]:
!python -m src.common.status --logs 5

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  backbone   subset                  images   shards tmp_files  상태
  clip       train                    61592    13/13         0  완료
  clip       val                       5080      2/2         0  완료
  dinov2     train                    61592    13/13         0  완료
  dinov2     val                       5080      2/2         0  완료
  dinov3     train                    61592    13/13         0  완료
  dinov3     train_limit1000           1000      1/1         0  완료
  dinov3     val                       5080      2/2         0  완료

== 예측: /content/drive/MyDrive/puppy.../preds ==
  dev: pred 11개, recon 11개, tmp_files 0
    RQ1-val-dinov2-proto-r1
    RQ1-val-dinov2-proto-r1-l2
    RQ1-val-dinov2-proto-r2
    RQ1-val-dinov2-proto-r2-l2
    RQ1-val-dinov2-proto-r3
    RQ1-val-dinov3-proto-r1
    RQ1-val-dinov3-proto-r1-l2
    RQ1-val-dinov3-proto-r2
    RQ1-val-dinov3-proto-r2-l2
    RQ1-val-dinov3-proto-r3
    RQ1-val-dinov3-pr

**셀 8. 첫 실행 확인 (1,000장 시간 재기)**: 샤드가 '완료'이고 `tmp_files`가 0인지(Drive에서 임시 파일 이름 바꾸기가 되는지) 보고, 장/초로 전체 시간을 추정한다.

In [9]:
!python -m src.features.extract_deep --backbone dinov3 --subset train --limit 1000 --tag timing
!python -m src.common.status --logs 1 --eta

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1030.44it/s]
[ok] 0장, 0.0초, 0.0장/초 → /content/drive/MyDrive/puppy.../features/dinov3/train_limit1000
== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  backbone   subset                  images   shards tmp_files  상태
  clip       train                    61592    13/13         0  완료
  clip       val                       5080      2/2         0  완료
  dinov2     train                    61592    13/13         0  완료
  dinov2     val                       5080      2/2         0  완료
  dinov3     train                    61592    13/13         0  완료
  dinov3     train_limit1000           1000      1/1         0  완료
  dinov3     val                       5080      2/2         0  완료

== 예측: /content/drive/MyDrive/puppy.../preds ==
  dev: pred 11개, recon 11개, tmp_files 0
    RQ1-val-dinov2-proto-r1
    RQ1-val-dinov2-proto-r1-l2
    RQ1-val-dinov2-proto-r2
    RQ1-val-dino

**셀 9. 특징 추출**: dinov3 → dinov2 → clip 순서로 train · val · eval. 끊기면 같은 셀을 다시 실행한다(끝난 샤드는 건너뜀).

In [10]:
for b in ["dinov3", "dinov2", "clip"]:
    for s in ["train", "val", "eval"]:
        !python -m src.features.extract_deep --backbone {b} --subset {s}

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1424.86it/s]
[ok] 0장, 0.0초, 0.0장/초 → /content/drive/MyDrive/puppy.../features/dinov3/train
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1385.34it/s]
[ok] 0장, 0.0초, 0.0장/초 → /content/drive/MyDrive/puppy.../features/dinov3/val
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1283.96it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/dog-mix-cv/src/features/extract_deep.py", line 121, in <module>
    main()
    ~~~~^^
  File "/content/dog-mix-cv/src/features/extract_deep.py", line 79, in main
    image_ids, loaders, index = build_items(args, paths)
                                ~~~~~~~~~~~^^^^^^^^^^^^^
  File "/content/dog-mix-cv/src/features/extrac

**셀 10. 평균점 섞기 (dev)**: 추출기 4종 × L2 없음/있음. 이미 있는 예측은 건너뛴다. `sift`는 C의 특징이 올라온 뒤에 돌아간다.

In [11]:
for b in ["dinov3", "dinov2", "clip", "sift"]:
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --skip-existing --l2

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 247, in <module>
    main()
    ~~~~^^
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 235, in main
    info = run(args.backbone, r, args.split, args.pool, args.l2, args.frac, args.eval_subset, paths,
               args.train_subset, args.eval_kind, args.val_split)
  File "/content/dog-mix-cv/src/rq1/prototype_fcls.py", line 135, in run
    ids_ev, Xev, iev = load_features(feat_root, backbone, eval_subset, pool)
                       ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/dog-mix-cv/src/common/features_io.py", line 64, in load_features
    raise FileNotFoundError(f"특징 파일 없음: {d}")
FileNotFoundError: 특징 파일 없음: /content/drive/MyDrive/puppy.../features/dinov3/eval
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_modu

**셀 10-1. UF 초안 평가셋 확인**: UF 공개 자료(믹스견 97 + 순종 3)의 칸 사진 200장(개마다 얼굴 · 전신)과 인덱스가 Drive에 있는지 본다. 정식 평가셋(A의 `eval_index.csv`, D의 SAM 3 크롭)이 아닌 임시 초안이라 특징 이름은 `eval_uf_panels`다. 사진은 화면에 띄우지 않는다(UF 이용 조건: 교육용 저장만 허용).

In [18]:
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

try:
    # 구글 드라이브 시트에 접근하기 위한 사용자 인증
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    # 'eval_index_UF_panels_colab' 스프레드시트 열기
    sheet_name = 'eval_index_UF_panels_colab'
    print(f"구글 스프레드시트 '{sheet_name}'에서 데이터를 가져오는 중입니다...")
    sh = gc.open(sheet_name)
    worksheet = sh.get_worksheet(0)

    # 데이터를 판다스 데이터프레임으로 변환
    data = worksheet.get_all_values()
    df_sheet = pd.DataFrame(data[1:], columns=data[0])

    # 타겟 CSV 파일 경로에 저장
    target_csv_path = f"{DRIVE_DIR}/evalset/eval_index_UF_panels.csv"
    df_sheet.to_csv(target_csv_path, index=False, encoding='utf-8')
    print(f"[V] 성공적으로 스프레드시트를 다운로드하여 {target_csv_path} 경로에 CSV로 변환 저장했습니다!")
    print("이제 위에 있는 '셀 10-1 (uf_code_check)'을 다시 실행해 보세요!")
except Exception as e:
    print(f"[X] 변환 중 오류가 발생했습니다: {e}")
    print("만약 권한 에러나 시트를 찾을 수 없다는 에러가 발생하면, 구글 드라이브에서 'eval_index_UF_panels_colab' 파일 이름을 우클릭하고 '공유 -> 링크가 있는 모든 사용자'로 설정을 임시 변경해 보거나, 파일명에 오타가 없는지 확인해 주세요.")

구글 스프레드시트 'eval_index_UF_panels_colab'에서 데이터를 가져오는 중입니다...
[V] 성공적으로 스프레드시트를 다운로드하여 /content/drive/MyDrive/puppy.../evalset/eval_index_UF_panels.csv 경로에 CSV로 변환 저장했습니다!
이제 위에 있는 '셀 10-1 (uf_code_check)'을 다시 실행해 보세요!


In [19]:
import pandas as pd
UF_DIR = f"{DRIVE_DIR}/evalset"
UF_INDEX = f"{UF_DIR}/eval_index_UF_panels.csv"
UF_BACKBONES = ["dinov3", "dinov2", "clip"]   # 셀 7에서 train 특징이 '완료'인 추출기만 남긴다
idx = pd.read_csv(UF_INDEX)
print(f"칸 사진 {len(glob.glob(f'{UF_DIR}/raw/UF/panels/*.jpg'))}장 (기대 200) | 인덱스 {len(idx)}행, "
      f"개 {idx['dog_id'].nunique()}마리 | split {idx['split'].value_counts().to_dict()}")
for p in [idx["path"].iloc[0], idx["path"].iloc[-1]]:
    print("[있음]" if os.path.exists(p) else "[없음]", p)
missing = [p for p in idx["path"] if not os.path.exists(p)]
assert not missing, f"사진 {len(missing)}장을 못 찾음 → 인덱스의 path 앞부분이 DRIVE_DIR과 같은지 확인: {missing[:3]}"
for f in ["truth_UF.csv", "breed_map_UF_draft.csv"]:
    print("[있음]" if os.path.exists(f"{UF_DIR}/uf_draft/{f}") else "[없음]", f"{UF_DIR}/uf_draft/{f}")

칸 사진 200장 (기대 200) | 인덱스 200행, 개 100마리 | split {'hold': 160, 'dev': 40}
[있음] /content/drive/MyDrive/puppy.../evalset/raw/UF/panels/UF001_face.jpg
[있음] /content/drive/MyDrive/puppy.../evalset/raw/UF/panels/UF119_body.jpg
[없음] /content/drive/MyDrive/puppy.../evalset/uf_draft/truth_UF.csv
[없음] /content/drive/MyDrive/puppy.../evalset/uf_draft/breed_map_UF_draft.csv


**셀 10-2. UF 특징 추출**: 칸 사진 200장을 `eval_uf_panels`로 저장한다. 끊기면 같은 셀을 다시 실행한다.

In [20]:
for b in UF_BACKBONES:
    !python -m src.features.extract_deep --backbone {b} --subset eval --eval-index "{UF_INDEX}" --save-as eval_uf_panels

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 211/211 [00:00<00:00, 1183.44it/s]
dinov3/eval_uf_panels shard 1/1: 100% 2/2 [00:50<00:00, 25.50s/it]
[ok] 200장, 51.0초, 3.9장/초 → /content/drive/MyDrive/puppy.../features/dinov3/eval_uf_panels
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 223/223 [00:00<00:00, 1374.85it/s]
dinov2/eval_uf_panels shard 1/1: 100% 2/2 [00:01<00:00,  1.04it/s]
[ok] 200장, 1.9초, 102.8장/초 → /content/drive/MyDrive/puppy.../features/dinov2/eval_uf_panels
Loading weights: 100% 398/398 [00:00<00:00, 1460.40it/s]
clip/eval_uf_panels shard 1/1: 100% 2/2 [00:01<00:00,  1.10it/s]
[ok] 200장, 1.8초, 109.2장/초 → /content/drive/MyDrive/puppy.../features/clip/eval_uf_panels


**셀 10-3. UF 평균점 섞기 (dev)**: 임시 dev 20마리만 돌린다(hold는 예측하지 않는다). L2 없음/있음, 이미 있는 예측은 건너뛴다. 실행 이름은 `RQ1-eval_uf_panels-{추출기}-proto-r{반복}`.

In [21]:
for b in UF_BACKBONES:
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --eval-subset eval_uf_panels --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --repeats 1 2 3 --split dev --eval-subset eval_uf_panels --skip-existing --l2

{'run': 'RQ1-eval_uf_panels-dinov3-proto-r1', 'n_images': 40, 'n_dogs': 20, 'n_classes': 127, 'mean_recon_err': 0.8104, 'status': 'ok', 'seconds': 1.4, 'who': 'B', 'split': 'dev', 'frac': 0.8, 'l2': False, 'pool': 'cls'}
{'run': 'RQ1-eval_uf_panels-dinov3-proto-r2', 'n_images': 40, 'n_dogs': 20, 'n_classes': 127, 'mean_recon_err': 0.8113, 'status': 'ok', 'seconds': 1.2, 'who': 'B', 'split': 'dev', 'frac': 0.8, 'l2': False, 'pool': 'cls'}
{'run': 'RQ1-eval_uf_panels-dinov3-proto-r3', 'n_images': 40, 'n_dogs': 20, 'n_classes': 127, 'mean_recon_err': 0.8106, 'status': 'ok', 'seconds': 1.2, 'who': 'B', 'split': 'dev', 'frac': 0.8, 'l2': False, 'pool': 'cls'}
{'run': 'RQ1-eval_uf_panels-dinov3-proto-r1-l2', 'n_images': 40, 'n_dogs': 20, 'n_classes': 127, 'mean_recon_err': 0.8115, 'status': 'ok', 'seconds': 1.0, 'who': 'B', 'split': 'dev', 'frac': 0.8, 'l2': True, 'pool': 'cls'}
{'run': 'RQ1-eval_uf_panels-dinov3-proto-r2-l2', 'n_images': 40, 'n_dogs': 20, 'n_classes': 127, 'mean_recon_err':

**셀 10-4. UF 임시 채점 + 상태 확인**: 매핑 초안 기준의 임시 채점이다. 정식 채점이 아니며 A의 `eval.py`가 나오면 그것으로 바꾼다.

In [22]:
!python scripts/provisional_score_uf.py --uf-dir "{UF_DIR}/uf_draft" --eval-subset eval_uf_panels
!python -m src.common.status --logs 5

임시 채점: 매핑 초안 기준, 정식 채점 아님(A의 eval.py로 대체 예정)
예측: /content/drive/MyDrive/puppy.../preds/dev | 평가 subset: eval_uf_panels | 정답: /content/drive/MyDrive/puppy.../evalset/uf_draft/truth_UF.csv
정답 품종으로 보는 매핑: 정확
Traceback (most recent call last):
  File "/content/dog-mix-cv/scripts/provisional_score_uf.py", line 234, in <module>
    main()
    ~~~~^^
  File "/content/dog-mix-cv/scripts/provisional_score_uf.py", line 133, in main
    c2b = breed_of_class(uf / "breed_map_UF_draft.csv", allowed)
  File "/content/dog-mix-cv/scripts/provisional_score_uf.py", line 51, in breed_of_class
    m = pd.read_csv(map_csv, encoding="utf-8-sig")
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/read

**셀 10-5. 순종 보조 평가 (Tsinghua val, dev 절반)**: 순종 사진을 순종으로 답하는지 본다. 믹스견 평가와 따로 보고한다. 먼저 val을 클래스별로 반씩 나눈 임시 분할(`evalset/val_split_seed1.csv`, 이미 있으면 건너뜀)을 만들고, 그 dev 쪽 2,540장만 평가한다. 평균점은 train에서만 만든다. `{추출기}/train`과 `{추출기}/val` 특징이 '완료'인 추출기만 돌아간다.

In [23]:
!python scripts/make_val_split.py
for b in ["dinov3", "dinov2", "clip"]:
    !python -m src.rq1.prototype_fcls --backbone {b} --eval-kind purebred --repeats 1 2 3 --split dev --skip-existing
    !python -m src.rq1.prototype_fcls --backbone {b} --eval-kind purebred --repeats 1 2 3 --split dev --skip-existing --l2

val 5080장, 클래스 127개 | dev 2540장, test 2540장 | seed 1
클래스당 dev 장수 분포: {20: 127} | test 장수 분포: {20: 127}
[건너뜀] 이미 있음 → /content/drive/MyDrive/puppy.../evalset/val_split_seed1.csv  (내용이 지금 만든 것과 같음)
[순종 보조 평가] dinov3/val 중 /content/drive/MyDrive/puppy.../evalset/val_split_seed1.csv 의 dev 행. 정답은 사진으로 붙인 품종 라벨이며 DNA 가 아니다.
[건너뜀] RQ1-val-dinov3-proto-r1: 이미 있음 → /content/drive/MyDrive/puppy.../preds/dev/pred_RQ1-val-dinov3-proto-r1.csv
[건너뜀] RQ1-val-dinov3-proto-r2: 이미 있음 → /content/drive/MyDrive/puppy.../preds/dev/pred_RQ1-val-dinov3-proto-r2.csv
[건너뜀] RQ1-val-dinov3-proto-r3: 이미 있음 → /content/drive/MyDrive/puppy.../preds/dev/pred_RQ1-val-dinov3-proto-r3.csv
[순종 보조 평가] dinov3/val 중 /content/drive/MyDrive/puppy.../evalset/val_split_seed1.csv 의 dev 행. 정답은 사진으로 붙인 품종 라벨이며 DNA 가 아니다.
[건너뜀] RQ1-val-dinov3-proto-r1-l2: 이미 있음 → /content/drive/MyDrive/puppy.../preds/dev/pred_RQ1-val-dinov3-proto-r1-l2.csv
[건너뜀] RQ1-val-dinov3-proto-r2-l2: 이미 있음 → /content/drive/MyDrive/puppy.../preds/dev/pred_RQ1-v

**셀 10-6. 순종 임시 채점**: 정답은 사진에 붙은 품종 라벨이며 DNA가 아니다. 정식 채점이 아니다.

In [24]:
!python scripts/provisional_score_purebred.py --uf-dir "{DRIVE_DIR}/evalset/uf_draft"

Traceback (most recent call last):
  File "/content/dog-mix-cv/scripts/provisional_score_purebred.py", line 178, in <module>
    main()
    ~~~~^^
  File "/content/dog-mix-cv/scripts/provisional_score_purebred.py", line 79, in main
    c2b = class_to_breed(Path(args.uf_dir) / "breed_map_UF_draft.csv", allowed)
  File "/content/dog-mix-cv/scripts/provisional_score_purebred.py", line 46, in class_to_breed
    m = pd.read_csv(map_csv, encoding="utf-8-sig")
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/d

**셀 11. 마무리**: 최종 상태를 보고 Drive에 쓰기를 끝낸 뒤 연결을 끊는다.

In [25]:
!python -m src.common.status
from google.colab import drive; drive.flush_and_unmount()

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  backbone   subset                  images   shards tmp_files  상태
  clip       eval_uf_panels             200      1/1         0  완료
  clip       train                    61592    13/13         0  완료
  clip       val                       5080      2/2         0  완료
  dinov2     eval_uf_panels             200      1/1         0  완료
  dinov2     train                    61592    13/13         0  완료
  dinov2     val                       5080      2/2         0  완료
  dinov3     eval_uf_panels             200      1/1         0  완료
  dinov3     train                    61592    13/13         0  완료
  dinov3     train_limit1000           1000      1/1         0  완료
  dinov3     val                       5080      2/2         0  완료

== 예측: /content/drive/MyDrive/puppy.../preds ==
  dev: pred 36개, recon 36개, tmp_files 0
    RQ1-eval_uf_panels-clip-proto-r1
    RQ1-eval_uf_panels-clip-proto-r1-l2
    RQ1-eval_uf_panels-clip-proto-r

In [26]:
!python -m src.common.status --logs 10
from google.colab import drive; drive.flush_and_unmount()

== 특징: /content/drive/MyDrive/puppy.../features (샤드 5000장) ==
  (폴더 없음: 아직 추출한 적 없음)

== 예측: /content/drive/MyDrive/puppy.../preds ==
  (폴더 없음: 아직 실행한 적 없음)

== 로그: /content/drive/MyDrive/puppy.../logs/runs.jsonl ==
  (로그 없음)
Drive not mounted, so nothing to flush and unmount.
